# Fourier Coefficients of Splines Derivatives
The disks with a notch represent a complex value. The phase is suggested by the orientation of the notch. The modulus is suggested by the color density, with a dense <span style="color:#1f77b4">**blue**</span> color depicting a large modulus. As the modulus dwindles, the blue color fades into the white background. When the modulus becomes negligible, the phase becomes indefinite and the color turns to <span style="color:#d3d3d3">**light gray**</span>.

*   The top row of complex numbers gives the first few Fourier coefficients of the gradient of a spline, as obtained through the proper adjustment of the Fourier coefficients $F\{f\}$ of the *spline* itself.
*   The bottom row of complex numbers gives the first few Fourier coefficients $F\{{\mathrm{D}}^{m}\{f\}\}$ of the *gradient of a spline*, as obtained directly.
*    The two rows do match perfectly, which provides a visual illustration of the mathematical equivalence between the two computational methods.

The spline $f$ of degree $n\in{\mathbb{N}}$ is shown in the <span style="color:#c20078">**fuchsia**</span> color. Its derivative of order $m\in[0\ldots n]$ is ${\mathrm{D}}^{m}\{f\}(x)=\frac{{\mathrm{d}}^{m}f(x)}{{\mathrm{d}}x^{m}}$ and is shown in the <span style="color:#2ca02c">**green**</span> color.

In [ ]:
# Load the required libraries
import cmath
import ipywidgets as widgets
import math
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal degree of the spline
max_delay = 2.0 # Maximal absolute delay
max_coeff = 15 # Maximal absolute Fourier index

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic splines with normal Gaussian coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 3)

# Plot
def update_plot (
    degree = 3,
    order = 2,
    period = 6,
    delay = 0.0
):
    global f

    # Update of the spline
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = f.degree
        )
    f.degree = degree
    f.delay = delay

    # Differentiation applied in the Fourier-series domain
    def jw (
        nu
    ):
        if 0 == order:
            return(1.0)
        return(((0 + 1j) * nu * 2.0 * np.pi / period) ** order)
    ff = np.array(
        [jw(nu) * f.fourier_coeff(nu) for nu in range(-max_coeff, max_coeff + 1)],
        dtype = complex
    )

    # Differentiation applied in the native (time or space) domain
    g = f.differentiated(order)
    fg = np.array(
        [g.fourier_coeff(nu) for nu in range(-max_coeff, max_coeff + 1)],
        dtype = complex
    )

    # Plots
    (fig, (ax1, ax2)) = plt.subplots(nrows = 2)
    r = 0.48
    # Caption
    ax1.text(
        -max_coeff,
        4.0 * r,
        r"$\left({{\mathrm{{j}}}}\,\nu\,\frac{{2\,\pi}}{{{}}}\right)^{{{}}}\,F\{{f\}}[\nu]$"
            .format(period, order)
    )
    ax1.text(
        -max_coeff,
        -5.0 * r,
        r"$F\{{{{\mathrm{{D}}}}^{{{}}}\{{f\}}\}}[\nu]$".format(order)
    )
    # Fourier coefficients
    ax1.set_xlim(left = -max_coeff, right = max_coeff)
    ax1.set_ylim(bottom = -2.0 * r - 0.1, top = 2.0 * r + 0.1)
    ax1.set_aspect(1.0)
    ax1.spines[:].set_color("None")
    ax1.tick_params(
        bottom = True,
        labelbottom = True,
        top = True,
        labeltop = False,
        left = False,
        labelleft = False
    )
    ax1.xaxis.set_major_locator(ticker.FixedLocator([-5, 0, 5, 10]))
    ax1.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    a0 = math.log(min(abs(ff)) + 0.01)
    a1 = math.log(max(abs(ff)) + 0.01)
    af = (np.log(abs(ff) + 0.01) - a0) / (a1 - a0) if a1 > a0 else np.ones(len(ff))
    ag = (np.log(abs(fg) + 0.01) - a0) / (a1 - a0) if a1 > a0 else np.ones(len(fg))
    for k1 in range(len(ff)):
        if cmath.isclose(
            0.0,
            ff[k1],
            rel_tol = math.sqrt(math.ulp(1.0)),
            abs_tol = math.sqrt(math.ulp(1.0))
        ):
            ax1.add_patch(patches.Circle((k1 - max_coeff, 0.55), r, color = "lightgray"))
        else:
            ax1.add_patch(patches.Wedge(
                (k1 - max_coeff, 0.55),
                r,
                cmath.phase(ff[k1]) * 180.0 / math.pi + 22.5,
                cmath.phase(ff[k1]) * 180.0 / math.pi - 22.5,
                color = "C0",
                alpha = min(max(0.0, af[k1]), 1.0)
            ))
    for k1 in range(len(fg)):
        if cmath.isclose(
            0.0,
            fg[k1],
            rel_tol = math.sqrt(math.ulp(1.0)),
            abs_tol = math.sqrt(math.ulp(1.0))
        ):
            ax1.add_patch(patches.Circle((k1 - max_coeff, -0.55), r, color = "lightgray"))
        else:
            ax1.add_patch(patches.Wedge(
                (k1 - max_coeff, -0.55),
                r,
                cmath.phase(fg[k1]) * 180.0 / math.pi + 22.5,
                cmath.phase(fg[k1]) * 180.0 / math.pi - 22.5,
                color = "C0",
                alpha = min(max(0.0, ag[k1]), 1.0)
            ))
    # Splines
    image = {f.image(), g.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))
    g.plot((fig, ax2), plotpoints = 201, plotrange = plotrange, curve_fmt = "-C2")
    f.plot((fig, ax2), plotpoints = 201, plotrange = plotrange, curve_fmt = "-m")
    plt.show()

# Interaction
degree_widget = widgets.IntSlider(min = 0, max = max_degree, value = 3)
order_widget = widgets.IntSlider(min = 0, max = 3, value = 2)
def update_order_range (
    *args
):
    order_widget.max = degree_widget.value
degree_widget.observe(update_order_range, "value")
widgets.interactive(
    update_plot,
    degree = degree_widget,
    order = order_widget,
    period = (1, max_period),
    delay = (-max_delay, max_delay)
)
